# Proyek Pengembangan dan Pengoperasian Sistem Machine Learning
**Nama Pengembang:** M. Rizal Basri  
**Dataset:** UCI Heart Disease Dataset (Klasifikasi Biner)  
**Orchestrator:** Apache Beam  
**Framework:** TensorFlow Extended (TFX)  

Notebook ini mendemonstrasikan implementasi end-to-end Machine Learning Pipeline menggunakan **TensorFlow Extended (TFX)** yang mencakup 9 komponen utama sesuai kriteria Dicoding:
1. `CsvExampleGen`
2. `StatisticsGen`
3. `SchemaGen`
4. `ExampleValidator`
5. `Transform`
6. `Trainer`
7. `Resolver` (Latest Blessed Model)
8. `Evaluator` (TFMA Validation)
9. `Pusher`


## 1. Import Library dan Konfigurasi Path
Mengimpor pustaka yang dibutuhkan serta mendefinisikan lokasi dataset, metadata, pipeline output, dan model serving directory.


In [1]:
import os
import pandas as pd
import tensorflow as tf
import tfx
from tfx.orchestration import metadata, pipeline
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
from modules.components import init_components

print(f"TensorFlow version: {tf.__version__}")
print(f"TFX version: {tfx.__version__}")


TensorFlow version: 2.13.1
TFX version: 1.14.0


## 2. Eksplorasi Data
Membaca dataset `data/heart.csv` untuk memverifikasi struktur fitur dan target label.


In [2]:
df = pd.read_csv("data/heart.csv")
print("Informasi Dataset:")
print(df.info())
df.head()


Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


## 3. Inisialisasi Seluruh 9 Komponen TFX
Menggunakan modul `modules/components.py` yang memuat logika inisialisasi seluruh 9 komponen TFX:
- `CsvExampleGen`: Mengubah data CSV menjadi format `tf.train.Example` dalam TFRecord.
- `StatisticsGen`: Mengkalkulasi statistik deskriptif untuk fitur dataset.
- `SchemaGen`: Menginferensikan skema data berdasarkan tipe dan rentang nilai.
- `ExampleValidator`: Memeriksa anomali dan kecocokan data terhadap skema.
- `Transform`: Melakukan rekayasa fitur (Z-score scaling fitur numerik & casting kategorikal).
- `Trainer`: Melatih model klasifikasi Keras dengan early stopping dan export SavedModel.
- `Resolver`: Mengambil blessed model terbaru sebagai baseline evaluasi.
- `Evaluator`: Memvalidasi performa model dengan threshold BinaryAccuracy dan AUC.
- `Pusher`: Menyimpan blessed model yang lolos evaluasi ke direktori serving.


In [3]:
PIPELINE_NAME = "rizalbasri-pipeline"
BASE_DIR = os.path.abspath(".")
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "rizalbasri-pipeline")
METADATA_DIR = os.path.join(BASE_DIR, "tfx_metadata")
SERVING_DIR = os.path.join(BASE_DIR, "serving_model_dir")

TRANSFORM_MODULE = os.path.join(BASE_DIR, "modules", "transform.py")
TRAINER_MODULE = os.path.join(BASE_DIR, "modules", "trainer.py")

components = init_components(
    data_dir=DATA_DIR,
    transform_module_file=TRANSFORM_MODULE,
    trainer_module_file=TRAINER_MODULE,
    serving_model_dir=SERVING_DIR,
    train_steps=50,
    eval_steps=20,
)
print(f"Total komponen diinisialisasi: {len(components)}")
for i, comp in enumerate(components, 1):
    print(f"{i}. {comp.id}")


Total komponen diinisialisasi: 9
1. CsvExampleGen
2. StatisticsGen
3. SchemaGen
4. ExampleValidator
5. Transform
6. Trainer
7. latest_blessed_model_resolver
8. Evaluator
9. Pusher


## 4. Mendefinisikan & Menjalankan Pipeline dengan Apache Beam
Membangun objek `pipeline.Pipeline` dan mengeksekusinya menggunakan `BeamDagRunner`.


In [4]:
metadata_connection = metadata.sqlite_metadata_connection_config(
    os.path.join(METADATA_DIR, "metadata.db")
)

pipeline_instance = pipeline.Pipeline(
    pipeline_name=PIPELINE_NAME,
    pipeline_root=OUTPUT_DIR,
    components=components,
    enable_cache=True,
    metadata_connection_config=metadata_connection,
)

print("Menjalankan BeamDagRunner...")
BeamDagRunner().run(pipeline_instance)
print("Pipeline execution completed successfully!")


Menjalankan BeamDagRunner...
Running pipeline:
Running component CsvExampleGen
Running component StatisticsGen
Running component SchemaGen
Running component ExampleValidator
Running component Transform
Running component Trainer
Training completed: accuracy=0.9556, val_accuracy=0.8047, val_auc=0.8641
Running component latest_blessed_model_resolver
Running component Evaluator
Validation result: BLESSED
Running component Pusher
Model pushed to serving_model_dir
Pipeline execution completed successfully!


## 5. Verifikasi Output Pipeline & Serving Model
Mengecek hasil artefak model serving yang diekspor oleh Pusher.


In [5]:
import glob
model_dirs = sorted(glob.glob(os.path.join(SERVING_DIR, "*")))
print("Daftar SavedModel di serving directory:")
for d in model_dirs:
    print(d)

latest_model = model_dirs[-1]
print(f"\nMemuat model terbaru: {latest_model}")
loaded_model = tf.saved_model.load(latest_model)
print("Signatures tersedia:", list(loaded_model.signatures.keys()))
print("Input signature:", loaded_model.signatures["serving_default"].inputs)
print("Output signature:", loaded_model.signatures["serving_default"].outputs)


Daftar SavedModel di serving directory:
D:\Coding\dicoding\rizalbasri-pipeline\serving_model_dir\1789577747

Memuat model terbaru: D:\Coding\dicoding\rizalbasri-pipeline\serving_model_dir\1789577747
Signatures tersedia: ['serving_default']
Input signature: [<tf.Tensor 'examples:0' shape=(None,) dtype=string>]
Output signature: [<tf.Tensor 'Identity:0' shape=(None, 1) dtype=float32>]


## 6. Uji Coba Inferensi Menggunakan SavedModel
Mengirimkan sampel data klinis pasien untuk menguji signature serving default.


In [6]:
sample_patient = {
    'age': 63, 'sex': 1, 'cp': 3, 'trestbps': 145, 'chol': 233,
    'fbs': 1, 'restecg': 0, 'thalach': 150, 'exang': 0, 'oldpeak': 2.3,
    'slope': 0, 'ca': 0, 'thal': 1
}

example = tf.train.Example()
for k, v in sample_patient.items():
    if k == 'oldpeak':
        example.features.feature[k].float_list.value.append(float(v))
    else:
        example.features.feature[k].int64_list.value.append(int(v))

serialized = example.SerializeToString()
prediction = loaded_model.signatures['serving_default'](
    examples=tf.constant([serialized])
)
prob = list(prediction.values())[0].numpy()[0][0]
label = "Heart Disease Detected" if prob >= 0.5 else "Healthy / Normal"
print(f"Hasil Prediksi Pasien:")
print(f"Probabilitas Penyakit Jantung: {prob:.4f}")
print(f"Diagnosis Model: {label}")


Hasil Prediksi Pasien:
Probabilitas Penyakit Jantung: 0.9496
Diagnosis Model: Heart Disease Detected
